# 06. Análisis y Modelos de Negocio

Este notebook utiliza el dataset consolidado para responder a las 5 preguntas estratégicas del Sunset Hospitality Group mediante estadística descriptiva y aprendizaje automático (Machine Learning).

**Preguntas a responder:**
1. Predicción del tipo de hotel (City vs. Resort) según el perfil de reserva.
2. Factores que influyen en la cancelación (Random Forest).
3. Clasificación de reservas de alto/bajo valor (Random Forest).
4. Perfiles de huéspedes recurrentes (K-Means).
5. Predicción de lealtad del huésped (Árbol de Decisión).

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, silhouette_score

# Configuración de rutas
PROCESSED_PATH = '../data/processed/'
RESULTS_PATH = '../data/results/'

# Carga de datos
df = pd.read_csv(os.path.join(PROCESSED_PATH, 'consolidated_bookings.csv'))
df_unified = pd.read_csv(os.path.join(PROCESSED_PATH, 'unified_dataset.csv'))

print(f"Dataset consolidado: {len(df)} registros")
print(f"Dataset unificado (pre-linkage): {len(df_unified)} registros")

## Pregunta 1: Predicción del Tipo de Hotel
¿Es posible predecir el tipo de hotel (City Hotel vs. Resort Hotel) que elegirá un huésped según las características de su reserva?

Comparamos tres algoritmos de clasificación (Regresión Logística, Árbol de Decisión y KNN) mediante GridSearchCV para encontrar la mejor configuración.

In [ ]:
# 1. Selección de variables disponibles al momento de la reserva
cat_cols_p1 = ['meal', 'market_segment', 'customer_type', 'deposit_type']
num_cols_p1 = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
               'adults', 'average_daily_rate', 'total_special_requests']

df_p1 = df[num_cols_p1 + cat_cols_p1 + ['hotel']].dropna()
df_p1 = pd.get_dummies(df_p1, columns=cat_cols_p1, drop_first=True)

X = df_p1.drop('hotel', axis=1)
le_hotel = LabelEncoder()
y = le_hotel.fit_transform(df_p1['hotel'])  # City Hotel=0, Resort Hotel=1

# 2. Partición: 15% dataset adicional de evaluación
X_temp, X_adicional, y_temp, y_adicional = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

scaler_p1 = StandardScaler()
X_train_scaled = scaler_p1.fit_transform(X_train)
X_adicional_scaled = scaler_p1.transform(X_adicional)

# 3. Configuración de modelos y Grid Search
modelos = {
    "Regresión Logística": {
        "modelo": LogisticRegression(random_state=42, max_iter=1000),
        "parametros": {'C': [0.1, 1.0, 10.0]}
    },
    "Árbol de Decisión": {
        "modelo": DecisionTreeClassifier(random_state=42),
        "parametros": {'max_depth': [5, 10, 15], 'min_samples_split': [2, 5]}
    },
    "KNN": {
        "modelo": KNeighborsClassifier(),
        "parametros": {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']}
    }
}

# 4. Entrenamiento, optimización y evaluación
resultados_p1 = []
print("Entrenando modelos para predicción del tipo de hotel...\n")

for nombre, config in modelos.items():
    print(f"-> Optimizando: {nombre}")
    clf = GridSearchCV(config["modelo"], config["parametros"], cv=3, scoring='accuracy', n_jobs=-1)
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_adicional_scaled)
    exactitud = accuracy_score(y_adicional, y_pred)
    resultados_p1.append({
        "Algoritmo": nombre,
        "Configuración Óptima": str(clf.best_params_),
        "Exactitud": exactitud
    })

df_resultados_p1 = pd.DataFrame(resultados_p1).sort_values(by="Exactitud", ascending=False)
exactitud_p1 = df_resultados_p1['Exactitud'].max()

print("\n--- Comparación de Algoritmos (Dataset Adicional) ---")
print(df_resultados_p1.to_string(index=False))

# 5. Visualización: Barras horizontales
fig, ax = plt.subplots(figsize=(8, 4))
colores = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax.barh(df_resultados_p1['Algoritmo'], df_resultados_p1['Exactitud'] * 100, color=colores)
ax.set_xlabel('Exactitud (%)')
ax.set_title('Pregunta 1: Comparación de Algoritmos\nPredicción del Tipo de Hotel')
ax.set_xlim(0, 100)
for bar in bars:
    width = bar.get_width()
    ax.text(width + 1, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', va='center')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, 'p1_comparacion_algoritmos.png'), dpi=150, bbox_inches='tight')
plt.show()

Iniciando el entrenamiento de modelos predictivos. Esto puede tomar unos minutos...

-> Optimizando y entrenando: Árbol de Decisión
-> Optimizando y entrenando: KNN (K-Nearest Neighbors)
-> Optimizando y entrenando: SVM (Support Vector Machine)

--- Tabla de Resultados Predictivos ---
                      Algoritmo                      Configuración Óptima  \
1     KNN (K-Nearest Neighbors)  {'n_neighbors': 3, 'weights': 'uniform'}   
0             Árbol de Decisión  {'max_depth': 5, 'min_samples_split': 5}   
2  SVM (Support Vector Machine)            {'C': 0.1, 'kernel': 'linear'}   

  Exactitud (Accuracy)  
1               91.11%  
0               88.89%  
2               88.89%  


## Pregunta 2: Factores de Cancelación
Entrenamos un Random Forest para identificar las variables que más impactan en `is_canceled`.

In [ ]:
# 1. Preparación de datos (One-Hot Encoding)
df_model_p2 = df.copy()
categorical_cols = ['hotel', 'meal', 'market_segment', 'deposit_type', 'customer_type']
num_features = ['lead_time', 'total_special_requests', 'average_daily_rate', 'booking_changes', 'previous_cancellations']

df_model_p2 = df_model_p2[num_features + categorical_cols + ['is_canceled']].dropna()
df_model_p2 = pd.get_dummies(df_model_p2, columns=categorical_cols, drop_first=True)

# 2. Separación de features y target
X = df_model_p2.drop('is_canceled', axis=1)
y = df_model_p2['is_canceled']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Entrenamiento del modelo
rf_p2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_p2.fit(X_train, y_train)

# 4. Evaluación
y_pred = rf_p2.predict(X_test)
exactitud_p2 = accuracy_score(y_test, y_pred)
print(f"Exactitud del Random Forest: {(exactitud_p2 * 100):.2f}%\n")

# 5. Importancia de variables
importances_p2 = pd.DataFrame({
    'Característica': X_train.columns,
    'Importancia': rf_p2.feature_importances_
}).sort_values(by='Importancia', ascending=False)

print("--- Top 10 Factores que Influyen en Cancelaciones ---")
print(importances_p2.head(10).to_string(index=False))

# 6. Visualización: Barras horizontales de importancia
fig, ax = plt.subplots(figsize=(10, 6))
top10 = importances_p2.head(10).iloc[::-1]  # invertir para que el más importante quede arriba
ax.barh(top10['Característica'], top10['Importancia'], color='#3498db')
ax.set_xlabel('Importancia')
ax.set_title('Pregunta 2: Top 10 Factores de Cancelación (Random Forest)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, 'p2_factores_cancelacion.png'), dpi=150, bbox_inches='tight')
plt.show()

Exactitud del modelo Random Forest: 84.13%

--- Top 10 Factores que más influyen en las cancelaciones ---
               Característica  Importancia
                    lead_time     0.310765
           average_daily_rate     0.285695
      deposit_type_Non Refund     0.134989
       total_special_requests     0.066446
       previous_cancellations     0.047671
              booking_changes     0.026027
     market_segment_Online TA     0.024459
        market_segment_Groups     0.018301
      customer_type_Transient     0.017947
customer_type_Transient-Party     0.017433


## Pregunta 3: Clasificación Alto/Bajo Valor
Definimos 'Alto Valor' como reservas por encima del percentil 75 de la tarifa diaria (`average_daily_rate`).

In [ ]:
# 1. Definición de la variable objetivo
umbral_valor = df['average_daily_rate'].quantile(0.75)
df['is_high_value'] = (df['average_daily_rate'] > umbral_valor).astype(int)

conteo_valor = df['is_high_value'].value_counts(normalize=True) * 100
print(f"Distribución de valor (Umbral P75 = {umbral_valor:.2f}):")
print(f"Bajo Valor: {conteo_valor[0]:.2f}% | Alto Valor: {conteo_valor[1]:.2f}%\n")

# 2. Preparación de datos predictivos
variables_iniciales = ['hotel', 'meal', 'market_segment', 'customer_type', 'deposit_type']
df_modelo3 = df[variables_iniciales + ['is_high_value']].dropna()
df_modelo3 = pd.get_dummies(df_modelo3, columns=variables_iniciales, drop_first=True)

X = df_modelo3.drop('is_high_value', axis=1)
y = df_modelo3['is_high_value']

# 3. Partición con dataset adicional de evaluación (15%)
X_temp, X_adicional, y_temp, y_adicional = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

# 4. Entrenamiento
rf_p3 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_p3.fit(X_train, y_train)

# 5. Evaluación sobre dataset adicional
y_pred_adicional = rf_p3.predict(X_adicional)
exactitud_p3 = accuracy_score(y_adicional, y_pred_adicional)

print("--- Evaluación Predictiva ---")
print(f"Exactitud sobre dataset adicional: {(exactitud_p3 * 100):.2f}%")

# 6. Visualización: Gráfica de pastel
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie([conteo_valor[0], conteo_valor[1]],
       labels=['Bajo Valor', 'Alto Valor'],
       autopct='%1.1f%%',
       colors=['#3498db', '#e74c3c'],
       startangle=90,
       explode=(0, 0.05))
ax.set_title(f'Pregunta 3: Distribución Alto/Bajo Valor\n(Umbral P75 = ${umbral_valor:.2f})')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, 'p3_distribucion_valor.png'), dpi=150, bbox_inches='tight')
plt.show()

Distribución de valor (Umbral P75 = 126.00):
Bajo Valor: 75.59% | Alto Valor: 24.41%

--- Evaluación Predictiva de la Pregunta 3 ---
Rendimiento del Random Forest sobre el dataset adicional de evaluación: 77.13%


## Pregunta 4: Perfiles de Huéspedes Recurrentes (Clustering)
Analizamos el comportamiento de los clientes que ya se han hospedado previamente.

In [ ]:
# 1. Filtrar huéspedes recurrentes
df_recurrent = df[df['is_repeated_guest'] == 1].copy()
cluster_features = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'total_special_requests']
X_clust = df_recurrent[cluster_features].dropna()

# 2. Partición: 15% como dataset adicional
X_train_clust, X_adicional_clust = train_test_split(X_clust, test_size=0.15, random_state=42)

# 3. Escalado
scaler_p4 = StandardScaler()
X_train_scaled = scaler_p4.fit_transform(X_train_clust)
X_adicional_scaled = scaler_p4.transform(X_adicional_clust)

# 4. Entrenamiento K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_train_scaled)

# 5. Evaluación con Coeficiente de Silueta
clusters_adicional = kmeans.predict(X_adicional_scaled)
sil_score_p4 = silhouette_score(X_adicional_scaled, clusters_adicional)

print("--- Evaluación del Clustering (K-Means) ---")
print(f"Coeficiente de Silueta (dataset adicional): {sil_score_p4:.4f}")
print("(Valores cercanos a 1 indican buena separación entre perfiles)\n")

# 6. Perfiles descubiertos
X_train_clust_labeled = X_train_clust.copy()
X_train_clust_labeled['Cluster'] = kmeans.labels_

print("--- Perfiles de Huéspedes Recurrentes (Promedios por Cluster) ---")
perfiles = X_train_clust_labeled.groupby('Cluster').mean()
print(perfiles.to_string())

# 7. Visualización: Scatter plot de clusters
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_train_scaled[:, 0], X_train_scaled[:, 2],
                     c=kmeans.labels_, cmap='viridis', alpha=0.6,
                     edgecolors='k', linewidths=0.3)
ax.set_xlabel('Lead Time (escalado)')
ax.set_ylabel('Noches entre Semana (escalado)')
ax.set_title('Pregunta 4: Clusters de Huéspedes Recurrentes')
plt.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, 'p4_clusters_recurrentes.png'), dpi=150, bbox_inches='tight')
plt.show()

--- Evaluación del Modelo de Clustering (K-Means) ---
Coeficiente de Silueta en el dataset adicional: 0.5593
(Nota: Valores cercanos a 1 indican una excelente separación entre perfiles)

--- Perfiles de Huéspedes Recurrentes (Promedios por Cluster) ---
          lead_time  stays_in_weekend_nights  stays_in_week_nights  total_special_requests
Cluster                                                                                   
0        116.951724                 2.744828              6.510345                1.606897
1          8.729084                 0.346082              1.178486                0.611554
2        273.446948                 0.216901              2.107981                0.169014


## Pregunta 5: Predicción de Lealtad
¿Es posible identificar, desde la primera reserva, si un huésped tiene el perfil para convertirse en cliente recurrente?

Se entrena un Árbol de Decisión para predecir `is_repeated_guest` basándose en variables disponibles al momento de la reserva.

In [ ]:
# 1. Preparación de datos
cat_cols_p5 = ['market_segment', 'customer_type', 'deposit_type']
num_cols_p5 = ['lead_time', 'average_daily_rate', 'total_special_requests']

df_modelo5 = df[num_cols_p5 + cat_cols_p5 + ['is_repeated_guest']].dropna()
df_modelo5 = pd.get_dummies(df_modelo5, columns=cat_cols_p5, drop_first=True)

X = df_modelo5.drop('is_repeated_guest', axis=1)
y = df_modelo5['is_repeated_guest']

# 2. Partición con dataset adicional (15%)
X_temp, X_adicional, y_temp, y_adicional = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

# 3. Entrenamiento del Árbol de Decisión
arbol_lealtad = DecisionTreeClassifier(max_depth=7, random_state=42)
arbol_lealtad.fit(X_train, y_train)

# 4. Evaluación sobre dataset adicional
y_pred_adicional = arbol_lealtad.predict(X_adicional)
exactitud_p5 = accuracy_score(y_adicional, y_pred_adicional)

print("--- Predicción de Lealtad ---")
print(f"Exactitud del Árbol de Decisión (dataset adicional): {(exactitud_p5 * 100):.2f}%")

# 5. Importancia de variables
importances_p5 = pd.DataFrame({
    'Característica': X_train.columns,
    'Importancia': arbol_lealtad.feature_importances_
}).sort_values(by='Importancia', ascending=False).head(10)

print("\n--- Top 10 Factores de Lealtad ---")
print(importances_p5.to_string(index=False))

# 6. Visualización: Barras verticales
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(importances_p5)), importances_p5['Importancia'], color='#2ecc71')
ax.set_xticks(range(len(importances_p5)))
ax.set_xticklabels(importances_p5['Característica'], rotation=45, ha='right')
ax.set_ylabel('Importancia')
ax.set_title('Pregunta 5: Factores que Predicen la Lealtad del Huésped')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, 'p5_factores_lealtad.png'), dpi=150, bbox_inches='tight')
plt.show()

--- Evaluación Predictiva de Lealtad (Pregunta 5) ---
Exactitud del Árbol de Decisión en el conjunto de datos adicional: 97.29%


## Guardar Resultados
Exportamos las métricas clave para el reporte final.

In [ ]:
results = {
    'metric': [
        'p1_mejor_exactitud_tipo_hotel',
        'p2_exactitud_cancelacion',
        'p3_exactitud_valor',
        'p3_umbral_alto_valor',
        'p4_silhouette_score',
        'p5_exactitud_lealtad'
    ],
    'value': [
        exactitud_p1,
        exactitud_p2,
        exactitud_p3,
        umbral_valor,
        sil_score_p4,
        exactitud_p5
    ]
}

pd.DataFrame(results).to_csv(os.path.join(RESULTS_PATH, 'model_results.csv'), index=False)
print("[OK] Resultados del análisis guardados en data/results/model_results.csv")

[OK] Resultados del análisis guardados en data/results/model_results.csv
